# Generate All Manuscript Figures

This notebook reproduces all figures currently used in `aa.tex`:

1. Main 10-panel SN plot grid (by syncing selected per-SN PNGs to `overleaf/img/`).
2. Absolute-magnitude scatter + marginals (`absmag_scatter_marginals_snr3.png`).
3. F275W-r color distribution (`f275w_r_color_distribution_snr3.png`).

It assumes pipeline products already exist in `outputs/plots/` and `outputs/sn_mag_summary_1kpc_full.csv`.

In [ ]:
from pathlib import Path
import re
import shutil
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
from matplotlib.lines import Line2D

mpl.rcParams.update({
    'font.family': 'DejaVu Serif',
    'mathtext.fontset': 'stix',
    'axes.unicode_minus': True,
})

In [ ]:
ROOT = Path.cwd().resolve()
OUTPUT_PLOTS = ROOT / 'outputs' / 'plots'
SUMMARY_CSV = ROOT / 'outputs' / 'sn_mag_summary_1kpc_full.csv'
PHOTOMETRY_CSV = ROOT / 'outputs' / 'photometry_summary_keep.csv'
OVERLEAF_IMG = ROOT / 'overleaf' / 'img'
OVERLEAF_IMG.mkdir(parents=True, exist_ok=True)

# The 10 panels currently shown in aa.tex (2 columns x 5 rows).
MAIN_GRID_FILES = [
    'ASASSN-13ch_plot.png',
    'ASASSN-13cj_plot.png',
    'ASASSN-13cp_plot.png',
    'ASASSN-13cu_plot.png',
    'ASASSN-14ba_plot.png',
    'ASASSN-14cu_plot.png',
    'ASASSN-14db_plot.png',
    'ASASSN-14dd_plot.png',
    'ASASSN-14dz_plot.png',
    'ASASSN-14fa_plot.png',
]

print(f'ROOT: {ROOT}')
print(f'Using summary table: {SUMMARY_CSV}')
print(f'Using photometry table: {PHOTOMETRY_CSV}')
print(f'Overleaf image dir: {OVERLEAF_IMG}')

In [ ]:
def sync_main_grid_images(source_dir: Path, target_dir: Path, image_names: list[str]) -> None:
    missing = []
    copied = 0
    for name in image_names:
        src = source_dir / name
        dst = target_dir / name
        if not src.exists():
            missing.append(name)
            continue
        shutil.copy2(src, dst)
        copied += 1

    print(f'Copied {copied} panel images to {target_dir}')
    if missing:
        print('Missing panel images:')
        for m in missing:
            print(f'  - {m}')

sync_main_grid_images(OUTPUT_PLOTS, OVERLEAF_IMG, MAIN_GRID_FILES)

In [ ]:
def parse_mag_field(value: object):
    s = str(value).strip()
    if s.lower() in {'nan', ''}:
        return np.nan, np.nan, 'nan'

    # Detection format: mag (err)
    m_det = re.match(r'^([+-]?\d+(?:\.\d+)?)\s*\(([+-]?\d+(?:\.\d+)?)\)$', s)
    if m_det:
        return float(m_det.group(1)), float(m_det.group(2)), 'det'

    # Upper-limit formats: (maglim) or LaTeX/ASCII bounds such as $<2.7$ or <2.7
    m_ul = re.match(r'^\(([+-]?\d+(?:\.\d+)?)\)$', s)
    if m_ul:
        return float(m_ul.group(1)), np.nan, 'ul'

    s_clean = s.replace('$', '').strip()
    m_ul_bound = re.match(r'^[<>]\s*([+-]?\d+(?:\.\d+)?)$', s_clean)
    if m_ul_bound:
        return float(m_ul_bound.group(1)), np.nan, 'ul'

    # Plain numeric fallback
    m_plain = re.match(r'^([+-]?\d+(?:\.\d+)?)$', s)
    if m_plain:
        return float(m_plain.group(1)), np.nan, 'det'

    return np.nan, np.nan, 'nan'


def load_absmag_arrays(csv_path: Path):
    df = pd.read_csv(csv_path)

    x_vals, x_errs, x_typ = [], [], []   # M_r
    y_vals, y_errs, y_typ = [], [], []   # M_F275W

    for _, row in df.iterrows():
        yv, ye, yt = parse_mag_field(row['UV abs mag'])
        xv, xe, xt = parse_mag_field(row['Adopted r abs mag'])
        x_vals.append(xv); x_errs.append(xe); x_typ.append(xt)
        y_vals.append(yv); y_errs.append(ye); y_typ.append(yt)

    x = np.asarray(x_vals, dtype=float)
    y = np.asarray(y_vals, dtype=float)
    xerr = np.asarray(x_errs, dtype=float)
    yerr = np.asarray(y_errs, dtype=float)
    xt = np.asarray(x_typ)
    yt = np.asarray(y_typ)

    return df, x, y, xerr, yerr, xt, yt


def plot_absmag_scatter_marginals(csv_path: Path, out_main: Path, out_alias: Path | None = None):
    df, x, y, xerr, yerr, xt, yt = load_absmag_arrays(csv_path)

    valid = np.isfinite(x) & np.isfinite(y)
    det_det = valid & (xt == 'det') & (yt == 'det')
    r_ul_uv_det = valid & (xt == 'ul') & (yt == 'det')
    r_det_uv_ul = valid & (xt == 'det') & (yt == 'ul')
    both_ul = valid & (xt == 'ul') & (yt == 'ul')

    x_all = x[valid]
    y_all = y[valid]
    x_margin = 0.65
    y_margin = 0.65
    x_left = np.nanmax(x_all) + x_margin
    x_right = np.nanmin(x_all) - x_margin
    y_bottom = np.nanmax(y_all) + y_margin
    y_top = np.nanmin(y_all) - y_margin

    fig = plt.figure(figsize=(8.3, 7.6))
    gs = GridSpec(2, 2, figure=fig, height_ratios=[1.0, 3.2], width_ratios=[3.2, 1.0], hspace=0.04, wspace=0.04)
    ax_top = fig.add_subplot(gs[0, 0])
    ax = fig.add_subplot(gs[1, 0])
    ax_right_hist = fig.add_subplot(gs[1, 1], sharey=ax)

    # Colorblind-safe palette
    col_det = '#2E86B7'
    col_ul = '#D55E00'
    col_histx = '#E3A21A'
    col_histy = '#2FAE8F'

    x_for_hist = x[valid & (xt == 'det')]
    y_for_hist = y[valid & (yt == 'det')]

    if x_for_hist.size:
        ax_top.hist(x_for_hist, bins=24, color=col_histx, edgecolor='white', linewidth=0.35)
    if y_for_hist.size:
        ax_right_hist.hist(y_for_hist, bins=24, orientation='horizontal', color=col_histy, edgecolor='white', linewidth=0.35)

    if np.any(det_det):
        ax.errorbar(
            x[det_det], y[det_det],
            xerr=xerr[det_det], yerr=yerr[det_det],
            fmt='o', ms=5.0, mfc=col_det, mec='white', mew=0.25,
            ecolor=col_det, elinewidth=0.8, alpha=0.9, capsize=0, zorder=3
        )

    ax.set_xlim(x_left, x_right)
    ax.set_ylim(y_bottom, y_top)

    x_span = abs(x_left - x_right)
    y_span = abs(y_bottom - y_top)
    dx = 0.030 * x_span
    dy = 0.030 * y_span
    padx = 0.015 * x_span
    pady = 0.015 * y_span

    def draw_x_ul(x0, y0):
        xl, xr = ax.get_xlim()
        sx = min(max(x0, xr + padx), xl - padx)
        ex = min(sx + dx, xl - padx)
        ax.annotate('', xy=(ex, y0), xytext=(sx, y0),
                    arrowprops=dict(arrowstyle='-|>', color=col_ul, lw=1.0, mutation_scale=8, shrinkA=0, shrinkB=0),
                    zorder=6, clip_on=False)

    def draw_y_ul(x0, y0):
        yb, ytlim = ax.get_ylim()
        sy = min(max(y0, ytlim + pady), yb - pady)
        ey = min(sy + dy, yb - pady)
        ax.annotate('', xy=(x0, ey), xytext=(x0, sy),
                    arrowprops=dict(arrowstyle='-|>', color=col_ul, lw=1.0, mutation_scale=8, shrinkA=0, shrinkB=0),
                    zorder=6, clip_on=False)

    def draw_diag_ul(x0, y0):
        xl, xr = ax.get_xlim()
        yb, ytlim = ax.get_ylim()
        sx = min(max(x0, xr + padx), xl - padx)
        sy = min(max(y0, ytlim + pady), yb - pady)
        ex = min(sx + dx, xl - padx)
        ey = min(sy + dy, yb - pady)
        ax.annotate('', xy=(ex, ey), xytext=(sx, sy),
                    arrowprops=dict(arrowstyle='-|>', color=col_ul, lw=1.0, mutation_scale=8, shrinkA=0, shrinkB=0),
                    zorder=6, clip_on=False)

    # Upper limits: arrows only (no base marker).
    for idx in np.where(r_ul_uv_det)[0]:
        xv, yv = x[idx], y[idx]
        if np.isfinite(yerr[idx]):
            ax.errorbar(xv, yv, yerr=yerr[idx], fmt='none', ecolor=col_ul, elinewidth=0.85, alpha=0.85, zorder=4)
        draw_x_ul(xv, yv)

    for idx in np.where(r_det_uv_ul)[0]:
        xv, yv = x[idx], y[idx]
        if np.isfinite(xerr[idx]):
            ax.errorbar(xv, yv, xerr=xerr[idx], fmt='none', ecolor=col_ul, elinewidth=0.85, alpha=0.85, zorder=4)
        draw_y_ul(xv, yv)

    for idx in np.where(both_ul)[0]:
        xv, yv = x[idx], y[idx]
        draw_diag_ul(xv, yv)

    # 1:1 reference line.
    xmin, xmax = sorted(ax.get_xlim())
    ymin, ymax = sorted(ax.get_ylim())
    d0, d1 = max(xmin, ymin), min(xmax, ymax)
    if d1 > d0:
        t = np.linspace(d0, d1, 200)
        ax.plot(t, t, ls='--', lw=1.2, color='0.35', alpha=0.85, zorder=2)

    if np.any(det_det):
        ax.axvline(np.nanmedian(x[det_det]), color='0.45', ls='--', lw=1.1, alpha=0.9)
        ax.axhline(np.nanmedian(y[det_det]), color='0.45', ls='--', lw=1.1, alpha=0.9)

    handles = [
        Line2D([0], [0], marker='o', linestyle='None', markersize=6, markerfacecolor=col_det, markeredgecolor='white', markeredgewidth=0.3, label='Detections'),
        Line2D([0], [0], linestyle='-', color=col_ul, lw=1.0, marker='>', markersize=5, markerfacecolor=col_ul, markeredgecolor=col_ul, label='Upper limits'),
    ]
    ax.legend(handles=handles, loc='upper left', frameon=True, framealpha=0.92, facecolor='white', edgecolor='0.8', fontsize=11)

    ax.set_xlabel(r'$M_r$ (mag)', fontsize=17)
    ax.set_ylabel(r'$M_{F275W}$ (mag)', fontsize=17)
    ax.grid(color='0.80', alpha=0.5, linewidth=0.6)

    ax_top.set_ylabel('N', fontsize=17)
    ax_right_hist.set_xlabel('N', fontsize=17)

    plt.setp(ax_top.get_xticklabels(), visible=False)
    plt.setp(ax_right_hist.get_yticklabels(), visible=False)

    for target_ax in (ax, ax_top, ax_right_hist):
        target_ax.tick_params(direction='in', top=True, right=True, labelsize=10)
        for s in target_ax.spines.values():
            s.set_linewidth(1.1)

    out_main.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(out_main, dpi=320, bbox_inches='tight')
    if out_alias is not None:
        fig.savefig(out_alias, dpi=320, bbox_inches='tight')
    plt.close(fig)

    n_det = int(np.sum(det_det))
    n_ul = int(np.sum(r_ul_uv_det | r_det_uv_ul | both_ul))
    print(f'Saved {out_main}  (n_det={n_det}, n_ul={n_ul})')
    if out_alias is not None:
        print(f'Saved {out_alias}')


def load_detected_colors(photometry_csv: Path):
    df = pd.read_csv(photometry_csv)

    uv_det = df['hst_1_is_upper_limit'].fillna(True).eq(False)
    r_det = df['rband_1_is_upper_limit'].fillna(True).eq(False)

    uv_mag = pd.to_numeric(df['WFC3_UVIS_F275W_1'], errors='coerce')
    r_mag = pd.to_numeric(df['rband_r_1'], errors='coerce')
    color = uv_mag - r_mag

    mask = uv_det & r_det & np.isfinite(color)
    return color[mask].to_numpy(dtype=float)


def plot_color_distribution(photometry_csv: Path, out_main: Path, out_alias: Path | None = None):
    color_vals = load_detected_colors(photometry_csv)

    fig, ax = plt.subplots(figsize=(6.0, 4.2))
    ax.hist(color_vals, bins=18, color='#56B4E9', edgecolor='white', linewidth=0.45)

    if color_vals.size:
        med = float(np.nanmedian(color_vals))
        ax.axvline(med, color='0.35', ls='--', lw=1.4)
        ax.text(0.98, 0.96, f'median={med:.2f}', transform=ax.transAxes, ha='right', va='top', fontsize=11)
        ax.text(0.98, 0.88, f'N={len(color_vals)}', transform=ax.transAxes, ha='right', va='top', fontsize=11)

    ax.set_xlabel('F275W-r (mag)', fontsize=15)
    ax.set_ylabel('N', fontsize=15)
    ax.grid(axis='y', color='0.86', linewidth=0.6)
    ax.tick_params(direction='in', top=True, right=True, labelsize=10)
    for s in ax.spines.values():
        s.set_linewidth(1.1)

    out_main.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(out_main, dpi=320, bbox_inches='tight')
    if out_alias is not None:
        fig.savefig(out_alias, dpi=320, bbox_inches='tight')
    plt.close(fig)

    print(f'Saved {out_main}')
    if out_alias is not None:
        print(f'Saved {out_alias}')

In [ ]:
plot_absmag_scatter_marginals(
    SUMMARY_CSV,
    OVERLEAF_IMG / 'absmag_scatter_marginals_snr3.png',
    OVERLEAF_IMG / 'absmag_scatter_marginals.png',
)

plot_color_distribution(
    PHOTOMETRY_CSV,
    OVERLEAF_IMG / 'f275w_r_color_distribution_snr3.png',
    OVERLEAF_IMG / 'f275w_r_color_distribution.png',
)

## Optional: regenerate per-SN panel plots from the full pipeline

If you need to recreate `outputs/plots/*.png` from scratch, run the photometry pipeline first (outside or inside this notebook), then rerun the sync cell above.

Example command:

```bash
MPLCONFIGDIR=/tmp/mpl conda run -n hostphot python scripts/hst_uv_photometry_pipeline.py --no-resume
```